# SPY 0DTE Flow-Following Strategy

Scan the 0DTE option trade tape for **large BUY-aggressor prints** (institutional 'smart money'). Copy the trade: buy the same option at the next 1-min bar's open, manage with PT / SL / time-stop. One trade/day, first qualifying print wins.

**Aggressor classification** (rough proxy without NBBO quotes):
- `trade_price >= bar_open` → buy aggressor → follow
- `trade_price <  bar_open` → sell aggressor → skip

**Caveats to keep in mind:**
- Multi-leg detection is NOT done. Some 'large buys' are one leg of a hedge — we'll trade them as if directional. Known noise source.
- Aggressor proxy is coarser than commercial flow services that use NBBO quotes.

**Sweep:** 4 thresholds × 4 PT × 3 SL × 3 TS = **144 configs**. First run will take 2–3 hours because Polygon needs to fetch trade tapes for ~10 ATM strikes × 252 days × ~paginated calls each.

## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Smoke test — one recent day
Default size threshold 1500 contracts. Confirms trade-tape fetch + aggressor classification + entry pipeline.

In [ ]:
!python -m iron_condor.cli --smoke -v

## 4. Full sweep — 144 configs over a year
**First run will take ~2 hours** because each day needs trade tapes for ~10 ATM-area strikes (paginated). Subsequent reruns are fast.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep

## 5. Top 20 configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 6. Size-threshold breakdown — does bigger = better?
Per-threshold average return, win rate, trade count.
Useful to see if 'bigger prints = stronger signal' actually holds in the data.

In [ ]:
import re
rows = []
for _, r in summary.iterrows():
    m = re.match(r'sz(\d+)\|pt(\d+)\|sl(\d+)\|ts(\d+)', r['config'])
    if m:
        rows.append({
            'sz': int(m.group(1)),
            'pt': int(m.group(2)),
            'sl': int(m.group(3)),
            'ts': int(m.group(4)),
            'return': r['total_return_pct'],
            'win_rate': r['win_rate'],
            'trades': r['n_trades'],
        })
grid = pd.DataFrame(rows)
print('=== Avg return % by size threshold ===')
print(grid.groupby('sz')['return'].agg(['mean', 'median', 'max']).round(1))
print('\n=== Avg win rate by size threshold ===')
print(grid.groupby('sz')['win_rate'].agg(['mean', 'median', 'max']).round(2))
print('\n=== Trade count by size threshold ===')
print(grid.groupby('sz')['trades'].agg(['mean', 'median', 'min', 'max']).round(0))

## 7. Top config breakdown — call vs put, exit reasons

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
top_cfg = summary.iloc[0]['config']
sub = trades[trades['config'] == top_cfg]
taken = sub[sub['exit_reason'].isin(['profit', 'stop', 'time_stop'])]
print(f'Top config: {top_cfg}')
print(f'Total days: {len(sub)}, taken trades: {len(taken)}')
print('\n=== Call vs put performance ===')
if not taken.empty and 'right' in taken.columns:
    print(taken.groupby('right').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2))
print('\n=== Exit reasons ===')
print(sub['exit_reason'].value_counts())
print('\n=== Print sizes captured (taken trades only) ===')
if not taken.empty and 'print_size' in taken.columns:
    print(taken['print_size'].describe().round(0))

## 8. Equity curve — top 3 configs

In [ ]:
import matplotlib.pyplot as plt
top3 = summary.head(3)['config'].tolist()
fig, ax = plt.subplots(figsize=(11, 5))
for cfg in top3:
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('Flow-following — top 3 equity curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()